# Cracks

## Objectives

This tutorial introduces the `crack` object in *Echoes*:

- the **crack density** parameterization (Budiansky–O'Connell);
- the syntax for open cracks and spring-interface cracks;
- multi-property cracks (simultaneous elasticity and conductivity).

## Imports

In [ ]:
import numpy as np
from echoes import *
import math

np.set_printoptions(precision=8, suppress=True)

## Crack density

An elliptical crack is geometrically defined as the limit of a flat ellipsoid whose smallest aspect ratio $\omega = c/a \to 0$ (see [§]). Because its volume fraction vanishes in this limit, the standard volume fraction is not a useful parameter. Instead, *Echoes* adopts the **Budiansky–O'Connell crack density** [@budiansky1976]:

$$
\varepsilon = n\,a^3
$$

where $n$ is the number density (number of cracks per unit volume) and $a$ the crack radius. For a family of penny-shaped cracks, the volume fraction is $\phi = \tfrac{4\pi}{3}\,\omega\,\varepsilon \to 0$ as $\omega \to 0$, confirming that density (not fraction) is the natural parameter.

For an elliptical crack of semi-axes $a \geq b \gg c$ ($\eta = b/a$, $\omega = c/a \ll 1$), the generalized crack density is $\varepsilon = n\,a^2 b = n\,\eta\,a^3$.

## Syntax

A `crack` object is built as:

```python
cr = crack(shape=spheroidal(ω, θ, φ),
           density=ε,
           symmetrize=[ISO],        # optional: isotropic distribution
           prop={"C": tZ4},         # open crack (zero stiffness)
           interf_prop={"C": [kn, kt]})  # spring interface (optional)
```

- `ω` is a small but strictly positive aspect ratio (e.g. $10^{-3}$), required for the numerical limit computation of the crack compliance (see [§]).
- `density` replaces `fraction` for crack phases: setting `density=ε` is equivalent to placing $n = \varepsilon / a^3$ cracks per unit volume.
- `symmetrize=[ISO]` projects the concentration tensor onto the isotropic class, representing a random (isotropic) distribution of crack orientations.
- `symmetrize=[TI, θ, φ]` projects the concentration tensor onto the transversely isotropic (TI) class with symmetry axis at polar angle $\theta$ and azimuthal angle $\phi$ (defaults: $\theta=\phi=0$, i.e. axis $\underline{e}_3$). This represents a distribution of crack orientations with TI symmetry — for instance, all crack normals lying in the plane perpendicular to the symmetry axis (see [§] for an example).

## Open crack vs. spring interface

- **Open crack** (traction-free faces): `prop={"C": tZ4}`. This is the default model; the crack faces exert no traction.
- **Spring interface** (partially bonded faces): add `interf_prop={"C": [kn, kt]}` where $k_n \geq 0$ is the normal spring stiffness and $k_t \geq 0$ the tangential one. The open crack ($k_n = k_t = 0$) and the perfectly bonded interface ($k_n, k_t \to \infty$) are the limiting cases. See [§] for the constitutive law.

The `density` attribute can be changed after construction:

```python
cr.density = 0.5
```

## Multi-property cracks

A `crack` object can carry simultaneous elastic and transport properties. For example, a fluid-filled crack with high in-plane conductance:

```python
cr = crack(shape=spheroidal(ω), density=ε,
           prop={"C": tZ4, "K": tensor(Kt, Kt, Kn)})
```

For the conductivity interface, `interf_prop={"K": [Kt1, Kt2, Kn]}` sets the conductances along the two tangential axes and the normal axis respectively; for a spheroidal crack, $K_{t1} = K_{t2} = K_t$ so the call reduces to `[Kt, Kt, Kn]`.

In [ ]:
ω = 1.e-3
Cs = stiff_Enu(1., 0.25)

cr = crack(shape=spheroidal(ω), density=0.3,
           prop={"C": tZ4})
print(cr)

In [ ]:
# With spring interface
cr_spr = crack(shape=spheroidal(ω), density=0.3,
               prop={"C": tZ4},
               interf_prop={"C": [5., 2.]})
print(cr_spr)

$\,$